# IMPORTS

In [1]:
import os
import gc
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


# CONFIGURATION

In [2]:
TRAIN_REPO = "OussamaBenSlama/qarai-guard-data-v0"
EVAL_REPO  = "OussamaBenSlama/qarai-guard-data-eval-v0"

MODEL_NAME = "asafaya/bert-mini-arabic"

HUB_MODEL_ID = "OussamaBenSlama/qarai-guard-12M-v0"

OUTPUT_DIR = "/kaggle/working/qarai-guard-bert-mini-arabic"

LABEL2ID = {
    "safe": 0,
    "safe_sensitive": 1,
    "harmful": 2,
    "privacy_pii": 3,
    "jailbreak": 4,
    "prompt_injection": 5,
    "misinformation": 6,
    "sexual_content": 7,
    "self_harm": 8,
    "hate_harassment": 9,
}

ID2LABEL = {
    v: k for k, v in LABEL2ID.items()
}

NUM_LABELS = len(LABEL2ID)

print("\nLabels:")
for k, v in LABEL2ID.items():
    print(f"{v}: {k}")


TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

#----------------------------------
MAX_LENGTH = 256
SEED = 42

NUM_EPOCHS = 2

LEARNING_RATE = 2e-5

WEIGHT_DECAY = 0.01

WARMUP_RATIO = 0.1


Labels:
0: safe
1: safe_sensitive
2: harmful
3: privacy_pii
4: jailbreak
5: prompt_injection
6: misinformation
7: sexual_content
8: self_harm
9: hate_harassment


In [3]:
set_seed(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
USE_BF16 = False
USE_FP16 = torch.cuda.is_available() and not USE_BF16

print("\nFP16:", USE_FP16)
print("BF16:", USE_BF16)


FP16: True
BF16: False


In [5]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    huggingface_token = user_secrets.get_secret("huggingface_token")
    print("Hugging Face token loaded from Kaggle secrets")

except Exception:
    print("Kaggle secrets not found. Using manual token input.")
    huggingface_token = input("Enter your Hugging Face token (hf_...): ").strip()

login(token=huggingface_token)

Hugging Face token loaded from Kaggle secrets


# LOAD DATA

In [6]:
print("\nLoading training dataset...")

train_raw = load_dataset(
    TRAIN_REPO,
    split="train"
)

print("Training examples:", len(train_raw))


print("\nLoading evaluation dataset...")

eval_raw = load_dataset(
    EVAL_REPO,
    split="train"
)

print("Evaluation examples:", len(eval_raw))


print("\nTrain columns:")
print(train_raw.column_names)

print("\nEval columns:")
print(eval_raw.column_names)


Loading training dataset...


README.md:   0%|          | 0.00/473 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 29.7MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/220887 [00:00<?, ? examples/s]

...

README.md:   0%|          | 0.00/467 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  960kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7315 [00:00<?, ? examples/s]

Evaluation examples: 7315

Train columns:
['text', 'label', 'language', 'source', 'original_label', '__index_level_0__']

Eval columns:
['text', 'label', 'language', 'source', 'original_label', '__index_level_0__']


In [7]:
train_ds = train_raw.select_columns([
    TEXT_COLUMN,
    LABEL_COLUMN,
])

eval_ds = eval_raw.select_columns([
    TEXT_COLUMN,
    LABEL_COLUMN,
])

# CLEAN DATA

In [8]:
def clean_example(example):

    text = example[TEXT_COLUMN]

    if text is None:
        text = ""

    text = str(text).strip()

    label = example[LABEL_COLUMN]

    if label is None:
        label = ""

    label = str(label).strip()

    return {
        "text": text,
        "label": label,
    }

In [9]:
train_ds = train_ds.map(
    clean_example,
    remove_columns=train_ds.column_names,
)

eval_ds = eval_ds.map(
    clean_example,
    remove_columns=eval_ds.column_names,
)

Map:   0%|          | 0/220887 [00:00<?, ? examples/s]

Map:   0%|          | 0/7315 [00:00<?, ? examples/s]

In [10]:
print(train_ds)

Dataset({
    features: ['text', 'label'],
    num_rows: 220887
})


In [11]:
print(eval_ds)

Dataset({
    features: ['text', 'label'],
    num_rows: 7315
})


In [12]:
train_ds = train_ds.filter(
    lambda x: len(x["text"]) > 0
)

eval_ds = eval_ds.filter(
    lambda x: len(x["text"]) > 0
)

print("\nAfter removing empty texts:")
print("Train:", len(train_ds))
print("Eval :", len(eval_ds))

Filter:   0%|          | 0/220887 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7315 [00:00<?, ? examples/s]


After removing empty texts:
Train: 220887
Eval : 7315


# CONVERT LABEL NAMES → INTEGER IDS

In [13]:
def encode_label(example):
    return {
        "label": LABEL2ID[example["label"]]
    }


train_ds = train_ds.map(encode_label)
eval_ds = eval_ds.map(encode_label)

Map:   0%|          | 0/220887 [00:00<?, ? examples/s]

Map:   0%|          | 0/7315 [00:00<?, ? examples/s]

#  TOKENIZATION

In [14]:
print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")


Loading tokenizer...


config.json:   0%|          | 0.00/509 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/334k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded.


In [15]:
def tokenize_function(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

In [16]:
print("\nTokenizing training data...")

train_tokenized = train_ds.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    remove_columns=["text"],
    desc="Tokenizing train",
)

print("\nTokenizing evaluation data...")

eval_tokenized = eval_ds.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    remove_columns=["text"],
    desc="Tokenizing eval",
)


Tokenizing training data...


Tokenizing train:   0%|          | 0/220887 [00:00<?, ? examples/s]


Tokenizing evaluation data...


Tokenizing eval:   0%|          | 0/7315 [00:00<?, ? examples/s]

# DATA COLLATOR

In [17]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)

# LOAD Base Model FOR CLASSIFICATION

In [18]:
print("\nLoading MARBERTv2...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

print("Model loaded.")

print(
    "Number of parameters:",
    sum(p.numel() for p in model.parameters()) / 1e6,
    "M"
)


Loading MARBERTv2...


model.safetensors: reconstructing file:   0%|          |  0.00B / 46.6MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: asafaya/bert-mini-arabic
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.
Number of parameters: 11.551498 M


# METRICS

In [19]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0,
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }

# TRAINING ARGUMENTS

In [20]:
training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    learning_rate=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,

    warmup_steps=4125,

    lr_scheduler_type="linear",


    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    gradient_accumulation_steps=1,


    fp16=USE_FP16,

    bf16=USE_BF16,



    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    greater_is_better=True,



    logging_strategy="steps",

    logging_steps=200,

    push_to_hub=False,  
    hub_model_id=HUB_MODEL_ID,

    report_to="none",


    

    dataloader_num_workers=2,

    dataloader_pin_memory=True,




    save_total_limit=3,


    seed=SEED,

    data_seed=SEED,
)


# TRAINER

In [21]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_tokenized,

    eval_dataset=eval_tokenized,

    data_collator=data_collator,

    processing_class=tokenizer,

    compute_metrics=compute_metrics,
)

In [22]:
print("\n" + "=" * 70)
print("STARTING TRAINING")
print("=" * 70)

train_result = trainer.train()


STARTING TRAINING


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.341058,0.819197,0.889405,0.916400,0.890585
2,0.199330,0.949894,0.895967,0.924704,0.896908


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
print("\nSaving best model...")

trainer.save_model(OUTPUT_DIR)

tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved to:", OUTPUT_DIR)


Saving best model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /kaggle/working/qarai-guard-bert-mini-arabic


# FINAL EVALUATION

In [24]:
print("\n" + "=" * 70)
print("FINAL EVALUATION")
print("=" * 70)

evaluation_results = trainer.evaluate()


FINAL EVALUATION


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
0.199330,0.949894,2,0.895967,0.924704,0.896908


In [25]:
for key, value in evaluation_results.items():

    if isinstance(value, float):

        print(
            f"{key}: {value:.4f}"
        )

    else:

        print(
            f"{key}: {value}"
        )

eval_loss: 0.9499
eval_accuracy: 0.8960
eval_macro_f1: 0.9247
eval_weighted_f1: 0.8969


In [26]:
print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)


predictions_output = trainer.predict(
    eval_tokenized
)

predictions = np.argmax(
    predictions_output.predictions,
    axis=-1
)

true_labels = predictions_output.label_ids


CLASSIFICATION REPORT


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [27]:
report = classification_report(
    true_labels,
    predictions,
    labels=list(range(NUM_LABELS)),
    target_names=[
        ID2LABEL[i]
        for i in range(NUM_LABELS)
    ],
    digits=4,
    zero_division=0,
)

print(report)

                  precision    recall  f1-score   support

            safe     0.7115    0.9521    0.8144      1189
  safe_sensitive     0.9518    0.9766    0.9641       728
         harmful     0.9370    0.7938    0.8594      2153
     privacy_pii     0.9606    0.7893    0.8665       617
       jailbreak     0.9562    0.9721    0.9641       359
prompt_injection     0.9837    0.9934    0.9885       304
  misinformation     0.9621    0.9416    0.9517       377
  sexual_content     0.9653    0.9804    0.9728       510
       self_harm     0.9337    0.9864    0.9593       514
 hate_harassment     0.9228    0.8901    0.9061       564

        accuracy                         0.8960      7315
       macro avg     0.9284    0.9276    0.9247      7315
    weighted avg     0.9086    0.8960    0.8969      7315



In [28]:
cm = confusion_matrix(
    true_labels,
    predictions,
    labels=list(range(NUM_LABELS))
)

cm_df = pd.DataFrame(
    cm,
    index=[
        ID2LABEL[i]
        for i in range(NUM_LABELS)
    ],
    columns=[
        ID2LABEL[i]
        for i in range(NUM_LABELS)
    ],
)

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

display(cm_df)


CONFUSION MATRIX


,safe,safe_sensitive,harmful,privacy_pii,jailbreak,prompt_injection,misinformation,sexual_content,self_harm,hate_harassment
safe,1132,7,33,2,2,2,2,4,3,2
safe_sensitive,4,711,9,0,1,0,3,0,0,0
harmful,320,26,1709,17,4,0,7,10,28,32
privacy_pii,69,0,49,487,8,0,0,2,1,1
jailbreak,1,0,3,0,349,3,0,0,2,1
prompt_injection,0,0,0,0,1,302,1,0,0,0
misinformation,6,3,7,1,0,0,355,0,2,3
sexual_content,6,0,1,0,0,0,0,500,0,3
self_harm,1,0,6,0,0,0,0,0,507,0
hate_harassment,52,0,7,0,0,0,1,2,0,502


In [29]:
print("\n" + "=" * 70)
print("TRAINING SUMMARY")
print("=" * 70)

print("Model:", MODEL_NAME)
print("Train examples:", len(train_ds))
print("Eval examples :", len(eval_ds))
print("Classes       :", NUM_LABELS)
print("Max length    :", MAX_LENGTH)
print("Epochs        :", NUM_EPOCHS)
print("Learning rate :", LEARNING_RATE)

print("\nBest checkpoint:")
print(trainer.state.best_model_checkpoint)

print("\nBest metric:")
print(trainer.state.best_metric)

print("\nFinal model saved to:")
print(OUTPUT_DIR)


TRAINING SUMMARY
Model: asafaya/bert-mini-arabic
Train examples: 220887
Eval examples : 7315
Classes       : 10
Max length    : 256
Epochs        : 2
Learning rate : 2e-05

Best checkpoint:
/kaggle/working/qarai-guard-bert-mini-arabic/checkpoint-13806

Best metric:
0.9247039172374032

Final model saved to:
/kaggle/working/qarai-guard-bert-mini-arabic


In [30]:
trainer.push_to_hub(
    commit_message="Upload version 0 selected by macro-F1"
)

print(f"Model uploaded: https://huggingface.co/{HUB_MODEL_ID}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model uploaded: https://huggingface.co/OussamaBenSlama/qarai-guard-12M-v0
